In [11]:
import math
import os
import re

import numpy as np
import pandas as pd
import networkx as nx
from sentence_transformers import SentenceTransformer


DEFAULT_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6388.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Общая реализация метода:

In [12]:
def normalize_rows(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def make_ngrams(text: str, n: int = 10, step: int = 10) -> list[str]:
    words = text.split()
    if not words:
        return [""]
    if len(words) <= n:
        return [" ".join(words)]
    return [" ".join(words[i:i + n]) for i in range(0, len(words) - n + 1, step)]


def csr(
    description_text: str,
    concept_title: str,
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
    concept_emb: np.ndarray | None = None,
    chunk_embs: np.ndarray | None = None,
) -> float:
    chunks = make_ngrams(description_text, n=n, step=step)

    if chunk_embs is None:
        chunk_embs = np.asarray(model.encode(chunks))
    if concept_emb is None:
        concept_emb = np.asarray(model.encode([concept_title]))[0]

    chunk_embs = normalize_rows(np.asarray(chunk_embs))
    concept_emb = concept_emb / max(np.linalg.norm(concept_emb), 1e-12)

    return float((chunk_embs @ concept_emb).sum())


def cer(
    description_text: str,
    concept_title: str,
    n: int = 10,
    step: int = 10,
    case_sensitive: bool = False,
) -> int:
    chunks = make_ngrams(description_text, n=n, step=step)

    if not case_sensitive:
        concept_title = concept_title.lower()
        chunks = [chunk.lower() for chunk in chunks]

    return sum(concept_title in chunk for chunk in chunks)


def transitive_reduce(g: nx.DiGraph) -> nx.DiGraph:
    if not nx.is_directed_acyclic_graph(g):
        return g
    reduced = nx.transitive_reduction(g)
    out = nx.DiGraph()
    out.add_nodes_from(g.nodes())
    out.add_edges_from(reduced.edges())
    return out


def ask_expert(topic_a: str, topic_b: str, score_ab: float, score_ba: float, prs: float, mode: str) -> int:
    print("\n" + "=" * 80)
    print(f"Пара: {topic_a!r}  <->  {topic_b!r}")
    print(f"{mode.upper()}({topic_a} -> {topic_b}) = {score_ab:.6f}")
    print(f"{mode.upper()}({topic_b} -> {topic_a}) = {score_ba:.6f}")
    print(f"PRS = {prs:.6f}")
    print("Введите решение:")
    print(f"  0 -> {topic_a} prerequisite для {topic_b}   ({topic_a} -> {topic_b})")
    print(f"  1 -> {topic_b} prerequisite для {topic_a}   ({topic_b} -> {topic_a})")
    print("  2 -> остановить разметку")

    while True:
        ans = input("Ваш выбор [0/1/2]: ").strip()
        if ans in {"0", "1", "2"}:
            return int(ans)


def prepare_embeddings(
    topic_to_text: dict[str, str],
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
):
    topic_embs = {
        topic: np.asarray(model.encode([topic]))[0]
        for topic in topic_to_text
    }

    chunk_embs = {
        topic: np.asarray(model.encode(make_ngrams(text, n=n, step=step)))
        for topic, text in topic_to_text.items()
    }

    return topic_embs, chunk_embs


def build_ranked_pairs(
    topic_to_text: dict[str, str],
    mode: str = "csr",
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
):
    topics = list(topic_to_text.keys())
    topic_embs, topic_chunk_embs = prepare_embeddings(topic_to_text, model=model, n=n, step=step)

    ranked = []
    pair_scores = {}

    for i in range(len(topics)):
        for j in range(i + 1, len(topics)):
            a, b = topics[i], topics[j]
            text_a, text_b = topic_to_text[a], topic_to_text[b]

            if mode == "csr":
                score_ab = csr(
                    description_text=text_a,
                    concept_title=b,
                    model=model,
                    n=n,
                    step=step,
                    concept_emb=topic_embs[b],
                    chunk_embs=topic_chunk_embs[a],
                )
                score_ba = csr(
                    description_text=text_b,
                    concept_title=a,
                    model=model,
                    n=n,
                    step=step,
                    concept_emb=topic_embs[a],
                    chunk_embs=topic_chunk_embs[b],
                )
            else:
                score_ab = cer(text_a, b, n=n, step=step)
                score_ba = cer(text_b, a, n=n, step=step)

            prs = max(score_ab, score_ba)
            ranked.append((prs, a, b))
            pair_scores[(a, b)] = (score_ab, score_ba)

    ranked.sort(reverse=True, key=lambda x: x[0])
    return ranked, pair_scores


def run_ace(
    topic_to_text: dict[str, str],
    mode: str = "csr",
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
    top_t_percent: float = 100.0,
):
    ranked_pairs, pair_scores = build_ranked_pairs(
        topic_to_text=topic_to_text,
        mode=mode,
        model=model,
        n=n,
        step=step,
    )

    g = nx.DiGraph()
    g.add_nodes_from(topic_to_text.keys())

    limit = math.ceil(len(ranked_pairs) * top_t_percent / 100)

    for prs, a, b in ranked_pairs[:limit]:
        if nx.has_path(g, a, b) or nx.has_path(g, b, a):
            continue

        score_ab, score_ba = pair_scores[(a, b)]
        ans = ask_expert(a, b, score_ab, score_ba, prs, mode)

        if ans == 2:
            break
        elif ans == 0:
            g.add_edge(a, b)
        else:
            g.add_edge(b, a)

        g = transitive_reduce(g)

    return g, ranked_pairs, pair_scores

# Функции для проведения тестирования и сравнения результатов метрики CSR на данных

In [13]:
def norm_name(s: str) -> str:
    return re.sub(r"\s+", "_", s.strip())


def csv_sort_key(file_name: str) -> tuple[int, str]:
    match = re.match(r"(\d+)-(\d+)\.csv$", file_name)
    if match:
        return int(match.group(1)), file_name
    return math.inf, file_name


def normalize_lookup_key(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", value.lower())


def build_description_index(descriptions_dir: str) -> dict[str, str]:
    index = {}
    for file_name in os.listdir(descriptions_dir):
        if not file_name.endswith('.txt'):
            continue

        stem = os.path.splitext(file_name)[0]
        key = normalize_lookup_key(stem)
        index[key] = os.path.join(descriptions_dir, file_name)

    return index


def resolve_description_path(concept: str, description_index: dict[str, str]) -> str | None:
    return description_index.get(normalize_lookup_key(concept))


def load_all_examples(dataset_dir: str, descriptions_dir: str):
    csv_files = sorted(
        [
            os.path.join(dataset_dir, file_name)
            for file_name in os.listdir(dataset_dir)
            if file_name.endswith(".csv")
        ],
        key=lambda path: csv_sort_key(os.path.basename(path)),
    )

    dataframes = []
    concepts = set()

    for csv_file in csv_files:
        df_part = pd.read_csv(csv_file)
        df_part.columns = [c.lower() for c in df_part.columns]
        df_part["source_file"] = os.path.basename(csv_file)

        dataframes.append(df_part)
        concepts.update(df_part["concept1"].astype(str))
        concepts.update(df_part["concept2"].astype(str))

    df = pd.concat(dataframes, ignore_index=True)

    description_index = build_description_index(descriptions_dir)
    missing_concepts = sorted(
        concept for concept in concepts
        if resolve_description_path(concept, description_index) is None
    )

    if missing_concepts:
        missing_set = set(missing_concepts)
        missing_mask = (
            df["concept1"].astype(str).isin(missing_set)
            | df["concept2"].astype(str).isin(missing_set)
        )
        skipped_rows = int(missing_mask.sum())
        df = df.loc[~missing_mask].reset_index(drop=True)

        print("Пропущены концепты без текстового описания:")
        for concept in missing_concepts:
            print(f"  - {concept}")
        print(f"Пропущено строк датасета: {skipped_rows}")
    else:
        skipped_rows = 0

    topic_to_text = {
        concept: open(
            resolve_description_path(concept, description_index),
            encoding="utf-8",
        ).read().strip()
        for concept in sorted(set(df["concept1"].astype(str)) | set(df["concept2"].astype(str)))
    }

    return df, topic_to_text, csv_files, missing_concepts, skipped_rows


def test_on_all_data(dataset_dir: str, descriptions_dir: str):
    df, topic_to_text, csv_files, missing_concepts, skipped_rows = load_all_examples(dataset_dir, descriptions_dir)

    topic_embs, chunk_embs = prepare_embeddings(
        topic_to_text=topic_to_text,
        model=DEFAULT_MODEL,
        n=10,
        step=10,
    )

    df["calculated_csr_score"] = [
        csr(
            description_text=topic_to_text[str(row["concept2"])],
            concept_title=str(row["concept1"]),
            model=DEFAULT_MODEL,
            n=10,
            step=10,
            concept_emb=topic_embs[str(row["concept1"])],
            chunk_embs=chunk_embs[str(row["concept2"])],
        )
        for _, row in df.iterrows()
    ]

    df["abs_diff"] = (df["calculated_csr_score"] - df["csr_score"]).abs()

    summary = (
        df.groupby("source_file", as_index=False)
        .agg(
            pair_count=("source_file", "size"),
            mean_abs_diff=("abs_diff", "mean"),
            max_abs_diff=("abs_diff", "max"),
        )
        .sort_values("source_file", key=lambda s: s.map(csv_sort_key))
        .reset_index(drop=True)
    )

    print(f"Количество файлов: {len(csv_files)}")
    print(f"Количество пар: {len(df)}")
    print(f"Количество уникальных концептов: {len(topic_to_text)}")
    print(f"Пропущено концептов без описания: {len(missing_concepts)}")
    print(f"Пропущено строк датасета: {skipped_rows}")
    print(f"Среднее абсолютное отклонение по всем данным: {df['abs_diff'].mean():.6f}")
    print(f"Максимальное абсолютное отклонение по всем данным: {df['abs_diff'].max():.6f}")

    return df, topic_to_text, summary

# Результаты сравнения на всех данных:

In [18]:
dataset_dir = r".\EKG-Dataset-main"
descriptions_dir = r".\EKG-Dataset-main\concept_descriptions"

df_test, topic_to_text, summary = test_on_all_data(dataset_dir, descriptions_dir)

print("\nСводка по каждому CSV-файлу:")
print(summary.to_string(index=False))


print("\nТоп-50 пар по абсолютному расхождению:")
df_test.nlargest(50, "abs_diff")[[
    "source_file",
    "concept1",
    "concept2",
    "csr_score",
    "calculated_csr_score",
    "abs_diff",
]]

Пропущены концепты без текстового описания:
  - 2D Pythagoras
Пропущено строк датасета: 1
Количество файлов: 10
Количество пар: 998
Количество уникальных концептов: 104
Пропущено концептов без описания: 1
Пропущено строк датасета: 1
Среднее абсолютное отклонение по всем данным: 0.481007
Максимальное абсолютное отклонение по всем данным: 13.171602

Сводка по каждому CSV-файлу:
 source_file  pair_count  mean_abs_diff  max_abs_diff
   0-100.csv         100       0.746397     13.171602
 100-200.csv          99       0.489325      2.453274
 200-300.csv         100       0.491056      2.737585
 300-400.csv         100       0.422334      1.952181
 400-500.csv         100       0.436454      4.015178
 500-600.csv         100       0.446979      3.801852
 600-700.csv         100       0.427687      2.327201
 700-800.csv         100       0.504260      6.677718
 800-900.csv         100       0.457304     12.477409
900-1000.csv          99       0.387416      7.564942

Топ-50 пар по абсолютному 

,source_file,concept1,concept2,csr_score,calculated_csr_score,abs_diff
16,0-100.csv,Time,"Speed, Distance, Time",21.9430,8.771398,13.171602
882,800-900.csv,Percentages of an Amount,Percentage Increase and Decrease,6.8354,19.312809,12.477409
25,0-100.csv,Time,Time Series and Line Graphs,20.7509,8.431622,12.319278
995,900-1000.csv,Fractions of an Amount,Percentage Increase and Decrease,5.9853,13.550242,7.564942
758,700-800.csv,Direct Proportion,Percentage Increase and Decrease,7.7538,14.431518,6.677718
784,700-800.csv,Writing Ratios,Percentage Increase and Decrease,7.5048,13.870052,6.365252
465,400-500.csv,Time,"Squares, Cubes, etc",10.1983,6.183122,4.015178
574,500-600.csv,Time,Length Scale Factors in Similar Shapes,9.0588,5.256948,3.801852
298,200-300.csv,"Rounding to the Nearest Whole (10, 100, etc)",Adding and Subtracting Negative Numbers,12.0918,9.354215,2.737585
110,100-200.csv,"Rounding to the Nearest Whole (10, 100, etc)",Fractions of an Amount,16.1509,13.697626,2.453274
